In [2]:
import torch
import math


# 注意力机制的实现
def attention(query, key, value, dropout=None):
    """

    :param query:  查询值矩阵
    :param key: 键值矩阵
    :param value:
    :param dropout:
    :return:
    """
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn



In [ ]:
import torch.nn as nn
import torch


# 多头注意力计算模块
class MultiHeadAttention(nn.Module):
    def __init__(self, args: ModelArgs, is_causal=False):
        super().__init__()
        assert args.dim % args.n_heads == 0

        # 模型并行处理大小
        model_parallel_size = 1

        self.n_local_model = args.n_heads // model_parallel_size

        # 每个头的维度
        self.head_dim = args.dim // args.n_heads

        self.wq = nn.Linear(args.dim, model_parallel_size * args.n_heads, bias=False)
        self.wk = nn.Linear(args.dim, model_parallel_size * args.n_heads, bias=False)
        self.wv = nn.Linear(args.dim, model_parallel_size * args.n_heads, bias=False)

        self.wo = nn.Linear(args.n_local_heads * self.head_dim, args.dim, bias=False)

        #  注意力的dropout
        self.attn_dropout = nn.Dropout(args.dropout)
        self.resid_dropout = nn.Dropout(args.rdropout)

        if is_causal:
            mask = torch.full((1, 1, args.max_seq_len, args.max_seq_len), -float('inf'))
            mask = torch.triu(mask, diagonal=1)
            self.requires_buffer("mask", mask)

        def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):
            bsz, seq_len, _ = q.shape
            xq, xk, xv = self.wq(q), self.wk(k), self.wv(v)
            xq = xq.view(bsz, seq_len, self.n_local_model, self.head_dim)
            xk = xk.view(bsz, seq_len, self.n_local_model, self.head_dim)
            xv = xv.view(bsz, seq_len, self.n_local_model, self.head_dim)

            xq = xq.transpose(1, 2)
            xk = xk.transpose(1, 2)
            xv = xv.transpose(1, 2)

            # 计算 QK^T / sqrt(d_k)，维度为 (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
            scores = torch.matmul(xq, xk.transpose(2, 3)) / math.sqrt(self.head_dim)

            if self.is_causal:
                assert hasattr(self, 'mask')
                scores = scores + self.mask[:, :, :seq_len, :seq_len]

            scores = F.softmax(scores.float(), dim=-1).type_as(xq)

            scores = self.attn_dropout(scores)
            # V * Score，维度为(B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
            output = torch.matmul(scores, xv)

            output = output.transpose(1, 2).contiguous().view(bsz, seq_len, -1)

            output = self.wo(output)
            output = self.resid_dropout(output)
            return output





